# Instant Gratification

## Introduction

This notebook builds a small web-browsing agent from first principles using ordinary Python and the OpenAI-compatible chat-completions API exposed by OpenRouter. The agent receives a task, asks a language model what to do, runs a Python tool when requested, and sends the tool result back to the model. It repeats that cycle until it can give a final answer or reaches its step limit.

The maze is the exercise, but the reusable idea is the **harness**: the code around a model that gives it tools, carries conversation state forward, and controls when it can act. The smallest useful mental model is:

```text
agent = model + tools + loop
```

### Learning goals

By the end, you should be able to:

- distinguish a tool's Python implementation from the JSON schema that describes it to a model;
- explain that a tool call is a request from the model, not an action the model performs by itself;
- follow the message history through a model response, tool execution, and the next model response;
- explain how a loop turns a single model call into a multi-step agent, and why the loop needs a limit.

### How to use this notebook

Run the cells from top to bottom. Later cells rely on names created earlier, such as `openrouter`, `tools`, and `handle_tool_call`. If you restart the kernel, run the setup cells again before trying the maze. The saved outputs are examples from earlier executions; a live model can choose different words, a different provider, or a different route through the maze.

The notebook makes external requests to OpenRouter and to the maze website. Keep your API key in a local `.env` file as `OPENROUTER_API_KEY=...`; never paste the key into a notebook cell or share it. The `openrouter/free` model alias may route different calls to different free models, so its answers and reliability can vary. Do not send confidential information to a free-model provider.

In [34]:
import json
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI

### Imports: what each dependency does

- `json` decodes the JSON arguments included in a model-requested tool call.
- `os` reads the API key from the process environment.
- `requests` makes the HTTP request to a maze page; Beautiful Soup turns the returned HTML into readable text.
- `load_dotenv` loads local environment settings from `.env`.
- `OpenAI` provides the client interface, while `display` and `Markdown` render model output in the notebook.

These imports set up the building blocks; they do not call the model or the website yet.

In [35]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key  and openrouter_api_key.startswith('sk-or-v1-'):
    print("OpenRouter API key is valid.")
else:
    print("OpenRouter API key is invalid.")


OpenRouter API key is valid.


### Load and inspect the API key

`load_dotenv(override=True)` reads values from `.env` into the process environment; with `override=True`, a value in that file takes precedence over an already-set environment value with the same name. `os.getenv` returns the key, or `None` if it is missing.

The `startswith('sk-or-v1-')` condition checks only that the key has the expected-looking prefix. It does **not** prove that the key is active, correctly copied, or authorized. The actual API request in the next section is the first real authentication check. The key is deliberately not printed.

In [36]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL ="openrouter/free"

# Retry up to 5 times when a free model is busy
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, 
                    api_key = openrouter_api_key,
                    timeout =120,
                    max_retries = 5)
messages = [{
    "role": "user",
    "content": "Hi, There!"
}]
response = openrouter.chat.completions.create(model = MODEL,
                                              messages=messages)
print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))


Answered by liquid/lfm-2.5-2.6b:free


Hey! Nice to meet you. How can I help today?

### First model call: verify the connection

`OpenAI(...)` creates a client that speaks the OpenAI chat-completions protocol but sends requests to OpenRouter's base URL. `MODEL` is set to `openrouter/free`, a routing alias rather than one fixed model. The response's `model` field reports which provider/model answered this particular request.

The request's `messages` is a list of role/content dictionaries. Here it contains only a user greeting. The client returns a response object; `choices[0].message.content` retrieves the first choice's text, and `Markdown` renders that text in the notebook. The configured timeout and retry count are useful for a free service that may be slow or temporarily busy; they do not guarantee that every request succeeds.

## Part 1 of the harness: the tool

A tool is an ordinary function in the host program plus a description of that function that the model can understand. The model can request `read_web_page`; Python, not the model, makes the HTTP request and returns the page text. This separation is fundamental: the model proposes an action, and the harness decides whether and how to execute it.

In [37]:
MAZE_URL = "https://maze.edwarddonner.com"

def read_web_page(url):
    """ 
    Read a web page in the maze and return the text on
    the page, without the HTML
    """
    print(f"Reading web page: {url}")
    if not url.startswith(MAZE_URL):
        return f"Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with {MAZE_URL}"
    response = requests.get(url, 
                           timeout = 20)
    page = BeautifulSoup(response.text,
                         "html.parser")   
    return page.body.get_text(separator="\n",
                              strip = True)

### The tool implementation: `read_web_page`

The function accepts one URL and returns text rather than raw HTML. It prints the requested URL for visibility, checks that the URL appears to belong to the maze, fetches it with a 20-second timeout, parses the response as HTML, and extracts the body text. `separator='\n'` keeps text from separate HTML elements readable; `strip=True` removes surrounding whitespace.

**Important limitation:** `url.startswith(MAZE_URL)` is a simple demonstration check, not a safe production allowlist. A URL such as `https://maze.edwarddonner.com.attacker.example/` also starts with that string. A production tool should parse the URL and compare its exact hostname (and scheme), and should consider redirects, HTTP errors, response size, and content type. `requests.get` follows redirects by default, and this example does not call `raise_for_status()`.

The timeout bounds how long this particular HTTP request waits. It does not limit response size or make page content trustworthy. Treat fetched page text as external data, even when the example site is benign.

In [38]:
print(read_web_page("https://maze.edwarddonner.com"))

Reading web page: https://maze.edwarddonner.com
The Maze
This page is for an AI that can read web pages. Your goal is to find the way out of a small maze and bring back the message that is waiting there. Every location in the maze is a web page that describes the room and lists its exits.
Step 1 is the entrance:
Location 1
https://maze.edwarddonner.com/1
Start with the maze now. Open the entrance, then follow the exits one page at a time. Some passages are dead ends, so go back and try another way when you reach one.
Keep the person you are working with updated as you go. After each location, say where you are and what you found there, and show a mini-map of the maze as you have seen it so far, drawn in text with the rooms you have visited and the exits you know about. When you reach the way out, give the message and a final map of your whole journey.
If you are a person reading this, give this address to ChatGPT or Claude and ask it to do what the page says.


### Exercise the allowed path

This call demonstrates the whole local tool function without involving the language model: Python fetches the maze's landing page, strips its HTML, and prints the resulting text. Notice that the page itself contains instructions for an AI. At this point they are just text returned by a website; they only influence the agent later when the model receives them as tool output.

In [39]:
print(read_web_page("https://pandagentic.ai"))

Reading web page: https://pandagentic.ai
Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with https://maze.edwarddonner.com


### Exercise the rejected path

This second call uses a URL outside the maze's apparent domain, so the function returns its explanatory message instead of requesting that page. This illustrates an important harness responsibility: do not assume a model's proposed tool arguments are safe merely because the model selected them. Validate inputs in the tool implementation.

The test demonstrates the intended behavior for an obviously different domain; it does not remove the prefix-check limitation described above.

In [40]:
read_web_page_json = {
    "type":"function",
    "function": {
        "name": "read_web_page",
        "description": " Read a web page and return the text on the page",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "The full URL of the web page to request"
                },
            },
            "required": ["url"],
        },
    },
}

tools = [read_web_page_json]

### Describe the tool to the model

`read_web_page_json` is the tool schema sent with a model request. Its outer `type` says this is a function tool. The function `name` must match the Python function the harness will run; the description tells the model when the function is useful; and the JSON Schema under `parameters` defines an object with a string property named `url`. Listing `url` in `required` means the model should supply it.

The schema does not execute Python and is not a security boundary. The model can still return malformed or unexpected arguments, so the host-side handler must parse and validate them. `tools` is a list because a model request can offer more than one tool.

In [41]:
def handle_tool_call(tool_call):
    """ 
    Run the tool that the model asks for and package 
    the result as a message.
    """
    try:
        arguments = json.loads(tool_call.function.arguments)
        result = read_web_page(**arguments)
    except Exception as error:
        result = f"The tool call failed: {error}"
    return {
        "role": "tool",
        "tool_call_id" : tool_call.id,
        "content": result
    }

### Translate a tool request into a tool result

`handle_tool_call` is the adapter between the API's tool-call object and the local Python function:

1. `tool_call.function.arguments` is a JSON string, so `json.loads` turns it into keyword arguments.
2. `read_web_page(**arguments)` invokes the Python implementation using those values.
3. The returned dictionary uses role `tool`, includes the matching `tool_call_id`, and puts the result in `content`.

That ID is how the API associates a tool's result with the particular request that asked for it. If execution raises an ordinary exception, the handler returns an explicit failure message as tool output so the model can see that the action failed. In a production application, failures should also be logged and classified, and validation should be more specific than a broad `except Exception`.

## Using the tool once

Before adding a loop, this section demonstrates one complete tool round trip. First the model receives a user request and the tool schema. If it chooses the tool, the response contains a tool-call object. The notebook then runs that tool and makes another model request with both the original conversation and the tool result.

In [42]:
messages = [{
    "role": "user",
    "content": "Please read https://maze.edwarddonner.com and tell me what it says. Just describe the page for now, without going onto the maze."
}]
response = openrouter.chat.completions.create(
    model = MODEL,
    messages = messages,
    tools= tools
)
message = response.choices[0].message

print(f"Answer by {response.model}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(message.tool_calls)

Answer by liquid/lfm-2.5-2.6b:free
Finish reason: tool_calls
[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-97aab01af088093f', function=Function(arguments='{"url": "https://maze.edwarddonner.com"}', name='read_web_page'), type='function', index=0)]


In [43]:
messages.append(message)
for tool_call in message.tool_calls:
    messages.append(handle_tool_call(tool_call))

response = openrouter.chat.completions.create(
                                    model = MODEL,
                                    messages = messages,
                                    tools = tools)
print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Reading web page: https://maze.edwarddonner.com
Answered by nvidia/nemotron-3-ultra-550b-a55b:free


The page at https://maze.edwarddonner.com is titled **"The Maze"** and serves as the introductory/instruction page for a web-based maze challenge designed for an AI to navigate.

Here's what the page says:

**Purpose:**  
It's a maze where each location is a separate web page that describes the room and lists its exits. The goal is to find the way out and retrieve a message waiting at the exit.

**Instructions for the AI:**
1. **Start at the entrance:** Location 1 – `https://maze.edwarddonner.com/1`
2. **Explore step by step:** Open each location page, read the room description, and follow the listed exits one page at a time.
3. **Handle dead ends:** If a passage leads to a dead end, go back and try another route.
4. **Keep the user updated:** After each location, report:
   - Where you are
   - What you found there
   - A mini-map (text-based) showing visited rooms and known exits so far
5. **Finish at the exit:** When you reach the way out, provide the final message and a complete map of the entire journey.

**Note for humans:**  
If a person is reading this page, they're instructed to give this address to an AI (like ChatGPT or Claude) and ask it to follow the maze instructions.

The page itself does not contain any maze content—it only sets up the challenge and points to the first location (`/1`).

### What the one-tool example demonstrates

The first request is allowed to use `tools`, so the model can return a structured request rather than a user-facing answer. The printed `finish_reason` and `message.tool_calls` help inspect that response. A `tool_calls` finish reason means the model is asking the client to run a tool; it does **not** mean the API ran the Python function.

The next cell appends the assistant's tool-call message to `messages`, runs every requested tool, and appends each matching tool result. It then sends that updated history to the model again. The model can now use the page text to produce a human-readable answer. This is the basic cycle the agent loop will repeat automatically.

## Part 2 of the harness: the loop

The tool makes the agent capable of acting; the loop lets it continue after each action. The conversation history is the agent's short-term state: each model request receives the prior user task, the assistant's tool requests, and the tool results accumulated so far.

In [44]:
SYSTEM_PROMPT = """ 
You are an agent that completes tasks by using your tools,
one step at a time. Your replies are displayed as Markdown,
so format them in Markdown with headings, bold text, and
lists. Put any map or diagram in a code block so that it 
lines up. Do not use emojis.
"""

### Set the agent's instructions

`SYSTEM_PROMPT` gives high-level behavior and formatting guidance that applies to this run. It asks for step-by-step tool use and Markdown output, including code blocks for maps so their alignment is preserved. The prompt shapes model behavior; it does not enforce tool permissions, guarantee that every instruction is followed, or validate the model's claims. Those responsibilities belong in the surrounding code and checks.

In [45]:
def run_agent(task, max_steps=25):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": task
        },
    ]

    for step in range(1, max_steps + 1):
        response = openrouter.chat.completions.create(
            model = MODEL,
            messages = messages,
            tools = tools
        )
        message = response.choices[0].message
        print(f"Step {step} answered by {response.model}")

        text = (message.content or "").strip()
        if text:
            display(Markdown(text))

        if message.tool_calls:
            messages.append(message)
            for tool_call in message.tool_calls:
                messages.append(handle_tool_call(tool_call))
        elif text:
            return
        else: 
            print("The reply was empty, so we ask again")
    print("The agent ran out of steps")


### Read the `run_agent` loop

The function turns the one-call pattern into a reusable agent:

1. **Start a fresh conversation.** `messages` begins with a system instruction and the user's task. It is a list because it represents the full conversation history; it must remain a list throughout the loop.
2. **Ask the model.** Each iteration submits that history and the available tools, then stores the returned assistant message in the separate variable `message`. Keeping `message` separate from `messages` matters: the former is one response, while the latter is the list to which future turns are appended.
3. **Show any text.** The model may include explanatory text with a tool request, so non-empty content is displayed as Markdown.
4. **Run requested tools.** For each tool call, the code first appends the assistant message containing the request and then appends the corresponding tool result. This preserves the order and the call IDs expected by the next API request. If there are multiple calls in one response, each gets a result.
5. **Stop or continue.** A response with text and no tool calls is treated as the final answer. A response with tool calls returns to the top of the loop, where the model sees the updated history. An empty response also continues, with a diagnostic message.
6. **Bound the run.** `range(1, max_steps + 1)` allows at most `max_steps` model requests, after which the function reports that it ran out of steps.

The step limit is a guard against an endless or unproductive tool cycle; it counts model requests, not necessarily individual tool calls. The function displays the answer in the notebook and returns no answer value to its caller. When you call it again, it creates a new conversation rather than reusing the previous run's history.

## Into the Maze

This final call gives the agent a broad task instead of a single page-reading request. The model must read the starting page, decide which location to visit, request the tool, interpret the returned page, and continue until it reports the exit message or reaches the step limit. The saved output below is one historical run, not a guaranteed transcript.

In [46]:
run_agent("Have a look at https://maze.edwarddonner.com and do what it says.")

Step 1 answered by dots-studio/dots-3-note-preview:free
Reading web page: https://maze.edwarddonner.com
Step 2 answered by nvidia/nemotron-3-super-120b-a12b:free
Reading web page: https://maze.edwarddonner.com/1
Step 3 answered by poolside/laguna-s-2.1:free


**Location 1** - Entrance. Exits: North -> 14, East -> 27.

### Mini-map (so far)
```
[1]* --N--> [14]
   |
   E   --> [27]
```

Starting exploration. Heading **North** to Location 14.

Reading web page: https://maze.edwarddonner.com/14
Step 4 answered by nvidia/nemotron-3-super-120b-a12b:free


**Location 14** - Long corridor with damp walls, tally scratched. Exits: North -> 62, South -> 1 (back).

### Mini-map (so far)
```
[1]* --N--> [14]* --N--> [62?]
   |
   E   --> [27]
```

Heading **North** to Location 62.

Reading web page: https://maze.edwarddonner.com/62
Step 5 answered by nvidia/nemotron-3-super-120b-a12b:free


...

**Location 83** - Round chamber with three ways out, cool air moving. Exits: North -> 25, East -> 56, West -> 48 (back).

### Mini-map (so far)
```
[1]* --N--> [14]* --N--> [62]* --E--> [48]* --E--> [83]* --N--> [25?]
   |                   |            |
   |                   E--> [48?]   E--> [56?]
   |
   E   --> [27]
```

Heading **North** to Location 25.

Reading web page: https://maze.edwarddonner.com/25
Step 10 answered by liquid/lfm-2.5-2.6b:free


**Location 25** - Stone staircase going up, daylight at the top. Exits: East -> 35, South -> 83 (back).

### Mini-map (so far)
```
[1]* --N--> [14]* --N--> [62]* --E--> [48]* --E--> [83]* --N--> [25]* --E--> [35?]
   |                   |            |
   |                   E--> [48?]   E--> [56?]
   |
   E   --> [27]
```

Heading **East** to Location 35.

Reading web page: https://maze.edwarddonner.com/35
Step 11 answered by nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free


**Message:**  
Welcome to the start of the course! Prepare for 3 intensive weeks ahead.

### Final Map of the Journey
```
[1] (entrance)
 |
N
 v
[14]
 |
N
 v
[62]
 |  \
 |   E → [48] → [83] → [25] → [35] (exit)
 |  /
 | /
N → [39] → [71] (dead end)
   |
   E → [48] (already visited)
   |
   E → [27] (unexplored)
```

- **Route taken:** `[1] → [14] → [62] → [48] → [83] → [25] → [35]` (exit)  
- **Dead end found:** `[71]` (reachable from `[39]`, which is north of `[62`); not traversed.  
- **Other unexplored exits:** `[27]` (east from `[1]`), `[39]` (north from `[62]`), `[56]` (east from `[83]`).

## Conclusion: the agent is the harness, not just the model

This notebook started with a plain chat request and progressively added the pieces that make the system agentic:

| Piece | What it contributes here |
|---|---|
| **Model** | Interprets the task, chooses whether to request a tool, and turns observations into the next step or final answer. |
| **Tool** | Gives the model a specific capability: read a maze page and return its text. |
| **Tool schema** | Communicates the tool's name, purpose, and expected arguments to the model. |
| **Harness / loop** | Maintains message history, executes requested calls, returns results with matching IDs, and stops after a final answer or a limit. |

The model does not directly browse the web in this example. It proposes a tool call; Python performs the request; the tool result becomes new context for the next model call. This division of responsibilities is useful beyond mazes: it is how an application can give an LLM bounded capabilities while keeping execution in code.

### Reference checklist

- Keep the conversation history (`messages`) separate from the latest response (`message`).
- Append the assistant tool-call message before its tool result, and preserve the tool-call ID.
- Treat the schema as a description, not as validation or authorization.
- Validate tool inputs in Python, set timeouts and limits, and handle failures visibly.
- Treat web content as untrusted input; the model may follow instructions found in it.
- Bound agent runs and expect model choice, wording, tool decisions, and results to vary.

### Troubleshooting and next experiments

- **Missing variable / `NameError`:** run the notebook from the top so the setup and function-definition cells have executed in the current kernel.
- **Authentication or request error:** check that the `.env` file is in the working directory used by the notebook, that `OPENROUTER_API_KEY` is set, and that the key is active. The prefix check alone is not enough.
- **Agent stops early or wanders:** free-model routing is variable. Try the task again, inspect the page/tool outputs, and consider a specific model or a suitable higher step limit. A larger limit also permits more API calls and does not guarantee success.
- **Tool-call failure:** inspect the printed URL and the tool result. Confirm the schema's function name and argument name match the Python handler, and that each tool result carries the ID of its call.

For further practice, trace `messages` after each iteration, lower `max_steps` to observe the stopping guard, and improve the URL validation before allowing arbitrary or user-supplied destinations. The larger lesson is to evaluate the whole system—model, tools, state, and control flow—not just the model's final prose.